<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Algorithmic Trading 

**Chapter 12 &mdash; FX Trading with FXCM (Server, Automation, Deployment)**

# This notebook is based on the fxcmpy.py module from FXCM. Unfortunately, FXCM has discontinued support and distribution of fxcmpy.py, so this notebook is no longer functional.

&copy; Dr Yves J Hilpisch | The Python Quants GmbH

http://tpq.io | [training@tpq.io](mailto:training@tpq.io) | [dyjh](http://twitter.com/dyjh)

<img src="https://hilpisch.com/pyalgo_cover_color.png" width="40%">

## Risk Disclaimer

<font size="-1">
Trading forex/CFDs on margin carries a high level of risk and may not be suitable for all investors as you could sustain losses in excess of deposits. Leverage can work against you. Due to the certain restrictions imposed by the local law and regulation, German resident retail client(s) could sustain a total loss of deposited funds but are not subject to subsequent payment obligations beyond the deposited funds. Be aware and fully understand all risks associated with the market and trading. Prior to trading any products, carefully consider your financial situation and experience level. Any opinions, news, research, analyses, prices, or other information is provided as general market commentary, and does not constitute investment advice. FXCM & TPQ will not accept liability for any loss or damage, including without limitation to, any loss of profit, which may arise directly or indirectly from use of or reliance on such information.
</font>

## Author Disclaimer

The author is neither an employee, agent nor representative of FXCM and is therefore acting independently. The opinions given are their own, constitute general market commentary, and do not constitute the opinion or advice of FXCM or any form of personal or investment advice. FXCM assumes no responsibility for any loss or damage, including but not limited to, any loss or gain arising out of the direct or indirect use of this or any other content. Trading forex/CFDs on margin carries a high level of risk and may not be suitable for all investors as you could sustain losses in excess of deposits.

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_core.git
import sys
sys.path.append('python_for_algo_trading_core')


In [1]:
import time
import numpy as np
import pandas as pd
import datetime as dt
from pylab import mpl, plt

In [2]:
plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'
%matplotlib inline

## Connecting to the API

In [3]:
import fxcmpy

ModuleNotFoundError: No module named 'fxcmpy'

In [ ]:
fxcmpy.__version__

In [ ]:
%time api = fxcmpy.fxcmpy(config_file='../../../data/pyalgo.cfg')

In [ ]:
instruments = api.get_instruments()

In [ ]:
print(instruments)

## Retrieving Historical Data

The parameter `period` must be one of `m1, m5, m15, m30, H1, H2, H3, H4, H6, H8, D1, W1` or `M1`.

In [ ]:
candles = api.get_candles('BTC/USD', period='m1', number=7500)

In [ ]:
candles

In [ ]:
candles['askclose'].plot(figsize=(10, 6));

## Deep Learning Strategy

In [ ]:
import tensorflow as tf
from keras.layers import Dense, Dropout
from keras.models import Sequential
from keras.regularizers import l1

In [ ]:
candles.head()

In [ ]:
symbol = 'BTC/USD'

In [ ]:
data = pd.DataFrame((candles['bidclose'] + candles['askclose']) / 2,
                    index=candles.index, columns=[symbol])

In [ ]:
data.info()

In [ ]:
data['r'] = np.log(data / data.shift(1))

In [ ]:
data['d'] = np.where(data['r'] > 0, 1, 0)

In [ ]:
window = 20

In [ ]:
data['sma'] = data[symbol].rolling(window).mean() 

In [ ]:
data['mom'] = data['r'].rolling(window).mean() 

In [ ]:
data['vol'] = data['r'].rolling(window).std() 

In [ ]:
data.dropna(inplace=True)

In [ ]:
data.head()

In [ ]:
features = ['r', 'sma', 'mom', 'vol']

In [ ]:
lags = 3

In [ ]:
def add_lags(data, lags=lags):
    cols = list()
    for f in features[:]:
        for lag in range(1, lags + 1):
            col = f'{f}_lag_{lag}'
            data[col] = data[f].shift(lag)
            cols.append(col)
    return data, cols

In [ ]:
data, cols = add_lags(data)

In [ ]:
data.dropna(inplace=True)

In [ ]:
# data.head()

In [ ]:
split = int(len(data) * 0.8)

In [ ]:
train = data.iloc[:split].copy()

In [ ]:
mu, std = train.mean(), train.std()

In [ ]:
train_ = (train - mu) / std

In [ ]:
# train_[cols].head()

In [ ]:
test = data.iloc[split:].copy()

In [ ]:
test_ = (test - mu) / std

In [ ]:
np.random.seed(100)
tf.random.set_seed(100)

In [ ]:
model = Sequential()
model.add(Dense(64, input_dim=len(cols), activation='relu'))
model.add(Dropout(rate=0.3, seed=100))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy',
              optimizer='adam', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
train['d'].value_counts()

In [ ]:
np.bincount(train['d'])

In [ ]:
%%time
model.fit(train_[cols], train['d'],
        epochs=75, verbose=False,
        validation_split=0.1, shuffle=False)

In [ ]:
model.evaluate(train_[cols], train['d'])

In [ ]:
model.evaluate(test_[cols], test['d'])

In [ ]:
test['p'] = np.where(model.predict(test_[cols]) > 0.5, 1, -1)

In [ ]:
test['p'].value_counts()

In [ ]:
test['s'] = test['p'] * test['r']

In [ ]:
test[['r', 's']].sum().apply(np.exp)

In [ ]:
test[['r', 's']].cumsum().apply(np.exp).plot(figsize=(10, 6));

In [ ]:
model.save('algorithm.keras')

In [ ]:
params = {'mu': mu, 'std': std, 'lags': lags}

In [ ]:
import pickle

In [ ]:
pickle.dump(params, open('params.pkl', 'wb'))

## Deploying the Strategy

In [ ]:
import keras

In [ ]:
algorithm = keras.models.load_model('algorithm.keras')

In [ ]:
test_[cols].iloc[-3:]

In [ ]:
np.where(model.predict(np.atleast_2d(test_[cols].iloc[3])) > 0.5, 1, -1)[0, 0]

In [ ]:
def prepare_features(resam):
    data = pd.DataFrame(resam['Mid'])
    data.columns = [symbol]
    data['r'] = np.log(data / data.shift(1))
    data['sma'] = data[symbol].rolling(3).mean()
    data['mom'] = data['r'].rolling(3).mean() 
    data['vol'] = data['r'].rolling(3).std()
    data, cols = add_lags(data)
    return data, cols

In [ ]:
order = api.create_market_buy_order('BTC/USD', 500)

In [ ]:
sel = ['tradeId', 'amountK', 'currency',
       'grossPL', 'isBuy']

In [ ]:
ticks = 0
position = 0
min_length = lags
def btc_dnn(data, df):
    global ticks
    global position
    global min_length
    ticks += 1
    print(ticks, end=' ')
    resam = df.resample('5s', label='right').last().ffill()
    resam['Mid'] = (resam['Bid'] + resam['Ask']) / 2
    data, cols = prepare_features(resam.iloc[:-1])
    #data.dropna(inplace=True)
    data_ = (data - mu) / std
    #print(data.tail())
    if len(data) > min_length:
        min_length += 1
        #print(data.tail())
        signal = np.where(algorithm.predict(
            np.atleast_2d(data_[cols].iloc[-1])) > 0.5, 1, -1)[0, 0]
        print(f'\n*** SIGNAL = {signal}')
        if position in [0, 1] and signal == -1:
            print('\n*** GOING SHORT***')
            order = api.create_market_sell_order('BTC/USD', (1 + position) * 500)
            api.get_open_positions()[sel]
            position = -1
        elif position in [0, -1] and signal == 1:
            print('\n*** GOING LONG ***')
            order = api.create_market_buy_order('BTC/USD', (1 - position) * 500)
            api.get_open_positions()[sel]
            position = 1

In [ ]:
api.subscribe_market_data('BTC/USD', (btc_dnn,))

In [ ]:
api.unsubscribe_market_data('BTC/USD')

## Deployment Script

Restart Jupyter Lab and run below only.

In [ ]:
#
# DNN Strategy for BTC/USD
# with FXCM
#
import time
import tensorflow as tf
import keras
import fxcmpy
import pickle
import numpy as np
import pandas as pd
import datetime as dt

algorithm = keras.models.load_model('algorithm.keras')
params = pickle.load(open('params.pkl', 'rb'))

ticks = 0
position = 0
min_length = params['lags']
symbol = 'BTC/USD'
features = ['r', 'sma', 'mom', 'vol']
sel = ['tradeId', 'amountK', 'currency',
       'grossPL', 'isBuy']

def add_lags(data, lags=params['lags']):
    cols = list()
    for f in features[:]:
        for lag in range(1, lags + 1):
            col = f'{f}_lag_{lag}'
            data[col] = data[f].shift(lag)
            cols.append(col)
    return data, cols

def prepare_features(resam):
    data = pd.DataFrame(resam['Mid'])
    data.columns = [symbol]
    data['r'] = np.log(data / data.shift(1))
    data['sma'] = data[symbol].rolling(3).mean()
    data['mom'] = data['r'].rolling(3).mean() 
    data['vol'] = data['r'].rolling(3).std()
    data, cols = add_lags(data)
    return data, cols


def btc_dnn(data, df):
    global ticks
    global position
    global min_length
    ticks += 1
    print(ticks, end=' ')
    resam = df.resample('5s', label='right').last().ffill()
    resam['Mid'] = (resam['Bid'] + resam['Ask']) / 2
    data, cols = prepare_features(resam.iloc[:-1])
    data_ = (data - params['mu']) / params['mu']
    if len(data) > min_length:
        min_length += 1
        signal = np.where(algorithm.predict(
            np.atleast_2d(data_[cols].iloc[-1])) > 0.5, 1, -1)[0, 0]
        print(f'\n*** SIGNAL = {signal}')
        if position in [0, 1] and signal == -1:
            print('\n*** GOING SHORT***')
            order = api.create_market_sell_order('BTC/USD', (1 + position) * 500)
            print(api.get_open_positions()[sel])
            position = -1
        elif position in [0, -1] and signal == 1:
            print('\n*** GOING LONG ***')
            order = api.create_market_buy_order('BTC/USD', (1 - position) * 500)
            print(api.get_open_positions()[sel])
            position = 1
            
api = fxcmpy.fxcmpy(config_file='pyalgo.cfg')
api.subscribe_market_data(symbol, (btc_dnn,))

In [ ]:
api.unsubscribe_market_data('BTC/USD')

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>